In [ ]:
# Cell 1: load overlay and configure shared parameters.
from time import sleep, time
import numpy as np
import matplotlib.pyplot as plt
from pynq import Overlay, MMIO, allocate

BITFILE = 'base_add.bit'
PL_CLK_HZ = 125_000_000
ADC_SAMPLE_HZ = 62_500_000
SENTINEL = np.uint32(0xDEADBEEF)

LED_BASE = 0x40000000
LED_RANGE = 0x1000
ADC_BASE = 0x40001000
ADC_RANGE = 0x1000
DMA_BASE = 0x40400000

# Board IO register offsets in led_ctrl_0.
LED_CTRL = 0x00
LED_VALUE = 0x08
LED_STATUS = 0x0C

# ADC capture register offsets in adc_capture_0.
CTRL = 0x00
STATUS = 0x04
SAMPLE_COUNT = 0x08
ADC_HALF = 0x0C
SAMPLE_DELAY = 0x10
DECIMATION = 0x14
CHANNEL_MASK = 0x18
CAPTURE_MODE = 0x1C
TRIGGER_MODE = 0x20
PRE_DELAY = 0x24
BUFFER_SELECT = 0x28
LATEST_A = 0x2C
LATEST_B = 0x30
SAMPLE_COUNTER = 0x34
FIFO_LEVEL = 0x38
ERROR_FLAGS = 0x3C
VERSION = 0x44
SAVED_COUNTER = 0x48
LAST_AXIS_WORD = 0x4C
DEBUG_STATE = 0x50
AXIS_SENT_COUNT = 0x54
AXIS_STALL_COUNT = 0x58
TLAST_COUNT = 0x5C
FIFO_BACKPRESSURE = 0x60
DROPPED_SAMPLE_COUNT = 0x64
CAPTURE_DONE_LATCHED = 0x68
CORE_DONE = 0x6C

S2MM_DMASR = 0x34

RGB_COLOR = {
    'OFF': 0b000,
    'RED': 0b001,
    'GREEN': 0b010,
    'BLUE': 0b100,
    'YELLOW': 0b011,
    'MAGENTA': 0b101,
    'CYAN': 0b110,
    'WHITE': 0b111,
}

COLOR_ORDER = ['OFF', 'RED', 'GREEN', 'BLUE', 'YELLOW', 'MAGENTA', 'CYAN', 'WHITE']

def encode_rgb(color):
    key = str(color).upper()
    if key not in RGB_COLOR:
        raise ValueError('Unknown RGB color %r' % color)
    return RGB_COLOR[key]

def find_dma(overlay):
    if hasattr(overlay, 'axi_dma_0') and hasattr(overlay.axi_dma_0, 'recvchannel'):
        return overlay.axi_dma_0
    for name in overlay.ip_dict:
        if 'dma' in name.lower():
            obj = getattr(overlay, name)
            if hasattr(obj, 'recvchannel'):
                return obj
    raise RuntimeError('Cannot find DMA object with recvchannel. IPs: %s' % list(overlay.ip_dict.keys()))

def set_board_io(led_mask=0, ld5_rgb=0, ld4_rgb=0):
    led_mask &= 0xF
    ld5_rgb &= 0x7
    ld4_rgb &= 0x7
    # Direct XDC/register mapping:
    # bits[3:0] -> LD0..LD3
    # bits[6:4] -> physical LD5 R/G/B = M15/L14/G14
    # bits[9:7] -> physical LD4 R/G/B = N15/G17/L15
    value = led_mask | (ld5_rgb << 4) | (ld4_rgb << 7)
    led_ip.write(LED_CTRL, 0x00)
    led_ip.write(LED_VALUE, value)
    return value

def set_rgb(led_mask=0, ld4='OFF', ld5='OFF'):
    return set_board_io(led_mask=led_mask, ld5_rgb=encode_rgb(ld5), ld4_rgb=encode_rgb(ld4))

def read_board_io():
    status = led_ip.read(LED_STATUS)
    return {
        'raw': status,
        'leds': status & 0xF,
        'ld4_register_field': (status >> 4) & 0x7,
        'ld5_register_field': (status >> 7) & 0x7,
        'buttons': (status >> 10) & 0xF,
    }


def pack_leds(ld0=False, ld1=False, ld2=False, ld3=False):
    # Direct XDC mapping: bit0->LD0, bit1->LD1, bit2->LD2, bit3->LD3.
    return ((1 if ld0 else 0) << 0) | ((1 if ld1 else 0) << 1) | ((1 if ld2 else 0) << 2) | ((1 if ld3 else 0) << 3)

def set_leds(ld0=False, ld1=False, ld2=False, ld3=False, ld4='OFF', ld5='OFF'):
    return set_rgb(
        led_mask=pack_leds(ld0, ld1, ld2, ld3),
        ld4=ld4,
        ld5=ld5,
    )

def read_buttons():
    raw = read_board_io()['buttons'] & 0xF
    # Direct XDC mapping: raw bit0->BTN0, bit1->BTN1, bit2->BTN2, bit3->BTN3.
    return {
        'raw': raw,
        'btn0': (raw >> 0) & 1,
        'btn1': (raw >> 1) & 1,
        'btn2': (raw >> 2) & 1,
        'btn3': (raw >> 3) & 1,
    }

def dump_capture_regs():
    print('STATUS          = 0x%08X' % adc_ip.read(STATUS))
    print('ERROR_FLAGS     = 0x%08X' % adc_ip.read(ERROR_FLAGS))
    print('LATEST_A/B      = %d / %d' % (adc_ip.read(LATEST_A), adc_ip.read(LATEST_B)))
    print('SAMPLE_COUNTER  = %d' % adc_ip.read(SAMPLE_COUNTER))
    print('FIFO_LEVEL      = %d' % adc_ip.read(FIFO_LEVEL))
    print('SAVED_COUNTER   = %d' % adc_ip.read(SAVED_COUNTER))
    print('LAST_AXIS_WORD  = 0x%08X' % adc_ip.read(LAST_AXIS_WORD))
    print('DEBUG_STATE     = %d' % adc_ip.read(DEBUG_STATE))
    print('AXIS_SENT_COUNT = %d' % adc_ip.read(AXIS_SENT_COUNT))
    print('TLAST_COUNT     = %d' % adc_ip.read(TLAST_COUNT))
    print('DROPPED_SAMPLES = %d' % adc_ip.read(DROPPED_SAMPLE_COUNT))
    print('VERSION         = 0x%08X' % adc_ip.read(VERSION))
    print('S2MM_DMASR      = 0x%08X' % dma.mmio.read(S2MM_DMASR))

def configure_capture(sample_count, adc_half_period, decimation, capture_mode, sample_delay=0):
    adc_ip.write(CTRL, 0x04)
    adc_ip.write(CTRL, 0x00)
    adc_ip.write(ERROR_FLAGS, 0xFFFFFFFF)
    adc_ip.write(SAMPLE_COUNT, int(sample_count))
    adc_ip.write(ADC_HALF, 1)       # legacy register; physical ADC clock is fixed at 62.5 MHz
    adc_ip.write(SAMPLE_DELAY, 0)   # legacy register; MMCM phase controls capture timing
    adc_ip.write(DECIMATION, int(decimation))
    adc_ip.write(CHANNEL_MASK, 0b11)
    adc_ip.write(CAPTURE_MODE, int(capture_mode))
    adc_ip.write(TRIGGER_MODE, 0)
    adc_ip.write(PRE_DELAY, 0)
    adc_ip.write(BUFFER_SELECT, 0)

def capture_once(sample_count, adc_half_period=1, decimation=1, capture_mode=2, sample_delay=0):
    sample_count = int(sample_count)
    buf = allocate(shape=(sample_count,), dtype=np.uint32)
    buf[:] = SENTINEL
    buf.flush()
    dma.recvchannel.transfer(buf)
    configure_capture(sample_count, adc_half_period, decimation, capture_mode, sample_delay)
    adc_ip.write(CTRL, 0x01)
    adc_ip.write(CTRL, 0x03)
    adc_ip.write(CTRL, 0x01)
    t0 = time()
    dma.recvchannel.wait()
    elapsed = time() - t0
    buf.invalidate()
    raw = np.array(buf, dtype=np.uint32)
    ch0 = raw & np.uint32(0x0FFF)
    ch1 = (raw >> np.uint32(16)) & np.uint32(0x0FFF)
    if np.any(raw == SENTINEL):
        raise RuntimeError('DMA buffer still contains sentinel values.')
    if adc_ip.read(AXIS_SENT_COUNT) != sample_count:
        raise RuntimeError('AXIS_SENT_COUNT mismatch: %d != %d' % (adc_ip.read(AXIS_SENT_COUNT), sample_count))
    if adc_ip.read(TLAST_COUNT) != 1:
        raise RuntimeError('TLAST_COUNT is not 1.')
    if adc_ip.read(DROPPED_SAMPLE_COUNT) != 0:
        raise RuntimeError('Dropped samples detected.')
    if adc_ip.read(STATUS) & (1 << 10):
        raise RuntimeError('STATUS.fatal_error is set.')
    return raw, ch0, ch1, elapsed

overlay = Overlay(BITFILE)
led_ip = MMIO(LED_BASE, LED_RANGE)
adc_ip = MMIO(ADC_BASE, ADC_RANGE)
dma = find_dma(overlay)

set_rgb(0, 'OFF', 'OFF')
print('Overlay loaded:', BITFILE)
print('led_ctrl_0    : MMIO(0x%08X, 0x%X)' % (LED_BASE, LED_RANGE))
print('adc_capture_0 : MMIO(0x%08X, 0x%X)' % (ADC_BASE, ADC_RANGE))
print('DMA object    :', dma)
print('IP dictionary :', list(overlay.ip_dict.keys()))
print('LED physical mapping: direct XDC, LD0->bit0, LD1->bit1, LD2->bit2, LD3->bit3')
print('BTN physical mapping: direct XDC, BTN0->bit0, BTN1->bit1, BTN2->bit2, BTN3->bit3')


In [ ]:
# Cell 2: test physical LEDs, physical RGB LEDs, and physical buttons.
print('Walking physical LD0..LD3')
for phys in range(4):
    kwargs = {'ld0': False, 'ld1': False, 'ld2': False, 'ld3': False}
    kwargs['ld%d' % phys] = True
    value = set_leds(**kwargs)
    sleep(0.5)
    print('Physical LD%d on, write=0x%03X status=%s' % (phys, value, read_board_io()))

print('\nTesting physical LD4 colors')
for color in COLOR_ORDER:
    value = set_leds(ld4=color, ld5='OFF')
    sleep(0.45)
    print('LD4 %-7s write=0x%03X status=%s' % (color, value, read_board_io()))

print('\nTesting physical LD5 colors')
for color in COLOR_ORDER:
    value = set_leds(ld4='OFF', ld5=color)
    sleep(0.45)
    print('LD5 %-7s write=0x%03X status=%s' % (color, value, read_board_io()))

print('\nVerified mixed-color example: LD0..LD3 on, physical LD4 blue, physical LD5 green')
value = set_leds(ld0=True, ld1=True, ld2=True, ld3=True, ld4='BLUE', ld5='GREEN')
sleep(1.0)
print('write=0x%03X status=%s buttons=%s' % (value, read_board_io(), read_buttons()))

print('\nLive physical button monitor for 15 seconds.')
print('Press BTN0, BTN1, BTN2, BTN3 one by one.')
t_end = time() + 15.0
last = None
while time() < t_end:
    buttons = read_buttons()
    if buttons['raw'] != last:
        print('raw=0b%s BTN0=%d BTN1=%d BTN2=%d BTN3=%d' % (
            format(buttons['raw'], '04b'),
            buttons['btn0'],
            buttons['btn1'],
            buttons['btn2'],
            buttons['btn3'],
        ))
        last = buttons['raw']
    sleep(0.03)

set_leds(False, False, False, False, ld4='OFF', ld5='OFF')
print('Board IO test done; outputs off.')


In [ ]:
# Cell 3: test fake stream -> AXIS FIFO -> AXI DMA -> DDR.
FAKE_SAMPLE_COUNT = 65536
raw, ch0, ch1, elapsed = capture_once(
    sample_count=FAKE_SAMPLE_COUNT,
    adc_half_period=1,
    decimation=1,
    capture_mode=2,
    sample_delay=0,
)

expected = np.arange(FAKE_SAMPLE_COUNT, dtype=np.uint32) & np.uint32(0x0FFF)
if not np.array_equal(ch0, expected):
    raise RuntimeError('CH0 fake pattern mismatch.')
if not np.array_equal(ch1, np.uint32(4095) - expected):
    raise RuntimeError('CH1 fake pattern mismatch.')

print('PASS: fake FIFO/DMA path')
print('elapsed = %.6f s' % elapsed)
print('CH0 first 16:', ch0[:16].tolist())
print('CH1 first 16:', ch1[:16].tolist())
dump_capture_regs()

plt.figure(figsize=(10, 4))
plt.plot(ch0[:256], label='fake CH0')
plt.plot(ch1[:256], label='fake CH1')
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
# Cell 4: test real ADC capture.
# For first bring-up, keep this moderate. Increase REAL_SAMPLE_COUNT after waveform looks sane.
REAL_SAMPLE_COUNT = 65536
REAL_ADC_HALF_PERIOD = 1       # compatibility argument; physical clock remains fixed at 62.5 MHz
REAL_DECIMATION = 3125         # 62.5 MSPS / 3125 = 20 kS/s
REAL_SAMPLE_DELAY = 0          # capture phase is fixed by MMCM

fs_hz = ADC_SAMPLE_HZ / REAL_DECIMATION
raw, ch0, ch1, elapsed = capture_once(
    sample_count=REAL_SAMPLE_COUNT,
    adc_half_period=REAL_ADC_HALF_PERIOD,
    decimation=REAL_DECIMATION,
    capture_mode=1,
    sample_delay=REAL_SAMPLE_DELAY,
)

print('PASS: real ADC DMA transfer completed')
print('elapsed      = %.6f s' % elapsed)
print('sample rate  = %.3f Hz' % fs_hz)
print('duration     = %.3f s' % (REAL_SAMPLE_COUNT / fs_hz))
print('CH0 mean/Vpp = %.2f / %d' % (float(ch0.mean()), int(ch0.max() - ch0.min())))
print('CH1 mean/Vpp = %.2f / %d' % (float(ch1.mean()), int(ch1.max() - ch1.min())))
dump_capture_regs()

n = min(1000, REAL_SAMPLE_COUNT)
t_ms = np.arange(n) / fs_hz * 1000.0
plt.figure(figsize=(12, 4))
plt.plot(t_ms, ch0[:n] - np.mean(ch0[:n]), label='CH0 DC removed')
plt.plot(t_ms, ch1[:n] - np.mean(ch1[:n]), label='CH1 DC removed', alpha=0.75)
plt.xlabel('Time (ms)')
plt.ylabel('ADC counts, DC removed')
plt.grid(True)
plt.legend()
plt.show()
